# Sudoku Ultra — Autoencoder Anti-Cheat Training Notebook

Trains a **sparse autoencoder** (PyTorch) on synthetic normal solve behaviour
to detect anomalous (bot/cheat-like) sessions at inference time.

| Component | Detail |
|-----------|--------|
| Architecture | Encoder 10→8→4→2, Decoder 2→4→8→10 |
| Bottleneck | 2-dimensional latent space |
| Sparsity | L1 regularisation on latent activations |
| Loss | MSE reconstruction |
| Threshold | mean_val_error + 2σ |
| Export | ONNX (opset 17) for fast inference |

**Feature vector (10-dim, all in [0,1])**:

| Idx | Name | Description |
|-----|------|-------------|
| 0 | time_mean_norm | Mean inter-fill time / 10 s |
| 1 | time_std_norm | Std-dev of fill times / 10 s |
| 2 | time_min_norm | Min fill time / 10 s |
| 3 | time_p10_norm | 10th-percentile fill time / 10 s |
| 4 | error_rate | errors / total attempts |
| 5 | hint_rate | hints / cells_to_fill |
| 6 | fill_rate_norm | cells/s normalised |
| 7 | duration_ratio | actual_ms / baseline_ms |
| 8 | completion_ratio | cells_filled / cells_to_fill |
| 9 | consistency_score | timing variance (uniform=low) |

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys, pathlib
ROOT = pathlib.Path("../../services/ml-service").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")

import numpy as np
import torch
print("PyTorch:", torch.__version__)
print("Python:", sys.version)

In [ ]:
# ── Inspect synthetic data ───────────────────────────────────────────────────
from app.ml.feature_extractor import (
    generate_normal_features,
    generate_anomalous_features,
    FEATURE_DIM,
)

X_normal = generate_normal_features(n=10_000)
X_anon   = generate_anomalous_features(n=500)

print(f"Normal shape:    {X_normal.shape}")
print(f"Anomalous shape: {X_anon.shape}")
print(f"\nNormal feature means:")
for i, v in enumerate(X_normal.mean(axis=0)):
    print(f"  f{i}: {v:.4f}")
print(f"\nAnomalous feature means:")
for i, v in enumerate(X_anon.mean(axis=0)):
    print(f"  f{i}: {v:.4f}")

In [ ]:
# ── Visualise feature distributions ─────────────────────────────────────────
import matplotlib.pyplot as plt

feature_names = [
    "time_mean_norm", "time_std_norm", "time_min_norm", "time_p10_norm",
    "error_rate", "hint_rate", "fill_rate_norm",
    "duration_ratio", "completion_ratio", "consistency_score",
]

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, (ax, name) in enumerate(zip(axes.flat, feature_names)):
    ax.hist(X_normal[:, i], bins=40, alpha=0.6, label="normal", color="steelblue")
    ax.hist(X_anon[:, i], bins=40, alpha=0.6, label="anomalous", color="tomato")
    ax.set_title(name, fontsize=8)
    ax.legend(fontsize=6)
plt.suptitle("Feature Distributions: Normal vs Anomalous", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Train autoencoder ────────────────────────────────────────────────────────
from app.ml.train_autoencoder import train_and_save

meta = train_and_save(
    n_normal=10_000,
    epochs=200,
    batch_size=256,
    lr=1e-3,
    sparsity_weight=1e-4,
    anomaly_threshold_sigma=2.0,
)

print("\nTraining complete:")
for k, v in meta.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# ── Evaluate reconstruction error distributions ──────────────────────────────
from app.ml.train_autoencoder import SparseAutoencoder, _mse
import pathlib, json

MODELS_DIR = pathlib.Path("../../services/ml-service").resolve().parent.parent / "ml/models"
model = SparseAutoencoder()
model.load_state_dict(torch.load(str(MODELS_DIR / "anomaly_autoencoder.pt"), map_location="cpu"))
model.eval()

threshold = meta["threshold"]

with torch.no_grad():
    recon_n, _ = model(torch.from_numpy(X_normal))
    errors_n = _mse(recon_n, torch.from_numpy(X_normal)).numpy()

    recon_a, _ = model(torch.from_numpy(X_anon))
    errors_a = _mse(recon_a, torch.from_numpy(X_anon)).numpy()

print(f"Normal   error — mean: {errors_n.mean():.6f}  max: {errors_n.max():.6f}")
print(f"Anomalous error — mean: {errors_a.mean():.6f}  max: {errors_a.max():.6f}")
print(f"Threshold: {threshold:.6f}")
print(f"\nDetection rate:      {(errors_a > threshold).mean():.1%}")
print(f"False positive rate: {(errors_n > threshold).mean():.1%}")

In [ ]:
# ── Plot reconstruction error distributions ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(errors_n, bins=60, alpha=0.7, label="Normal sessions", color="steelblue")
ax.hist(errors_a, bins=60, alpha=0.7, label="Anomalous sessions", color="tomato")
ax.axvline(threshold, color="black", linestyle="--", linewidth=2, label=f"Threshold ({threshold:.4f})")
ax.set_xlabel("Reconstruction Error (MSE)")
ax.set_ylabel("Count")
ax.set_title("Autoencoder Reconstruction Error Distribution")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise latent space ───────────────────────────────────────────────────
from sklearn.decomposition import PCA

with torch.no_grad():
    _, latent_n = model(torch.from_numpy(X_normal[:2000]))
    _, latent_a = model(torch.from_numpy(X_anon))

latent_n = latent_n.numpy()
latent_a = latent_a.numpy()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(latent_n[:, 0], latent_n[:, 1], alpha=0.3, s=10,
           label="Normal", color="steelblue")
ax.scatter(latent_a[:, 0], latent_a[:, 1], alpha=0.6, s=20,
           label="Anomalous", color="tomato", marker="x")
ax.set_title("Latent Space (2D) — Normal vs Anomalous")
ax.set_xlabel("z₁")
ax.set_ylabel("z₂")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Test AnomalyService API ──────────────────────────────────────────────────
from app.services.anomaly_service import anomaly_service

test_cases = [
    {"label": "Normal human",
     "time_elapsed_ms": 300_000, "cells_filled": 45, "errors_count": 3,
     "hints_used": 1, "difficulty": "medium"},
    {"label": "Suspicious (very fast, no errors)",
     "time_elapsed_ms": 8_000, "cells_filled": 50, "errors_count": 0,
     "hints_used": 0, "difficulty": "hard",
     "cell_fill_times_ms": [150] * 50},
    {"label": "Slow careful player",
     "time_elapsed_ms": 900_000, "cells_filled": 55, "errors_count": 8,
     "hints_used": 5, "difficulty": "expert"},
]

for tc in test_cases:
    label = tc.pop("label")
    result = anomaly_service.score(**tc)
    flag = "⚠ ANOMALOUS" if result["is_anomalous"] else "  normal"
    print(f"{flag}  {label}")
    print(f"   score={result['anomaly_score']:.4f}  error={result['reconstruction_error']:.6f}")

In [ ]:
# ── Verify ONNX model ────────────────────────────────────────────────────────
import onnxruntime as ort
import json

ort_session = ort.InferenceSession(str(MODELS_DIR / "anomaly_autoencoder.onnx"))

from app.ml.feature_extractor import extract_features
f = extract_features(
    time_elapsed_ms=300_000, cells_filled=45,
    errors_count=3, hints_used=1, difficulty="medium",
).reshape(1, -1)

outputs = ort_session.run(None, {"features": f})
recon_onnx = outputs[0].reshape(-1)
error_onnx = float(np.mean((f.reshape(-1) - recon_onnx) ** 2))

meta_loaded = json.loads((MODELS_DIR / "anomaly_meta.json").read_text())
print(f"ONNX reconstruction error: {error_onnx:.6f}")
print(f"Threshold:                 {meta_loaded['threshold']:.6f}")
print(f"Anomalous:                 {error_onnx > meta_loaded['threshold']}")
print("✓ ONNX inference working")

In [ ]:
# ── Inspect saved artefacts ──────────────────────────────────────────────────
for f in sorted(MODELS_DIR.glob("anomaly*")):
    size_kb = f.stat().st_size // 1024
    print(f"  {f.name}  ({size_kb} KB)")

print("\nMeta:")
meta_loaded = json.loads((MODELS_DIR / "anomaly_meta.json").read_text())
for k, v in meta_loaded.items():
    print(f"  {k}: {v}")